Uses GPU

In [2]:
import os
import cv2
import numpy as np
import cupy as cp  # Only requires: pip install cupy-cuda12x
import pandas as pd
from skimage import measure, morphology
from scipy.stats import skew, kurtosis
from cupyx.scipy.ndimage import uniform_filter, distance_transform_edt

# =========================
# GPU-ACCELERATED HELPERS
# =========================

def get_fractal_dim_cupy(img_gpu, thres):
    """Estimate fractal dimension using GPU-based box counting."""
    Z = img_gpu < thres
    h, w = Z.shape
    sizes = [2, 4, 8, 16, 32]
    counts = []
    for size in sizes:
        # Fast GPU max-pooling via reshape
        # This checks if ANY pixel in a 'size x size' block is active
        S = size
        res = Z[:h//S*S, :w//S*S].reshape(h//S, S, w//S, S).max(axis=(1, 3))
        counts.append(cp.sum(res > 0).get())
    
    coeffs = np.polyfit(np.log(sizes), np.log(counts), 1)
    return -coeffs[0]

# =========================
# PARAMETERS
# =========================
THRES = 100
CORE_THRES = 50
GRID_SIZE = 8
image_folder = r"./Data/"
results = []

for root, dirs, files in os.walk(image_folder):
    label = os.path.basename(root)
    
    for filename in files:
        if not filename.endswith(".bmp"): continue
        
        path = os.path.join(root, filename)
        img_cpu = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img_cpu = cv2.resize(img_cpu, (0, 0), fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
        
        # --- MOVE TO GPU ---
        img_gpu = cp.asarray(img_cpu)
        img_gpu_float = img_gpu.astype(cp.float32)

        # 1. Masking & Basic Math
        mask_low = img_gpu < THRES
        mask_high = img_gpu < CORE_THRES
        area_fraction = (cp.sum(mask_low) / img_gpu.size).get()

        # 2. Distance Transform (Thickness) - Accelerated via cupyx
        # This is a major bottleneck on CPU, much faster on GPU
        dist_map = distance_transform_edt(mask_low)
        mean_thickness = cp.mean(dist_map[mask_low]).get() if cp.any(mask_low) else 0
        max_thickness = cp.max(dist_map).get()

        # 3. FFT (Lightning Speed)
        f_shift = cp.fft.fftshift(cp.fft.fft2(img_gpu_float))
        fft_energy = cp.mean(cp.log(cp.abs(f_shift) + 1)).get()

        # 4. Local Texture (Variance Filter)
        # Var(X) = E[X^2] - (E[X])^2
        l_mean = uniform_filter(img_gpu_float, size=15)
        l_mean_sq = uniform_filter(cp.square(img_gpu_float), size=15)
        l_std = cp.sqrt(cp.maximum(0, l_mean_sq - cp.square(l_mean)))
        mean_l_std = cp.mean(l_std).get()
        std_l_std = cp.std(l_std).get()

        # 5. Fractal Dimension
        f_dim = get_fractal_dim_cupy(img_gpu, THRES)

        # 6. Grid Features (Vectorized on GPU)
        h, w = img_gpu.shape
        gh, gw = h // GRID_SIZE, w // GRID_SIZE
        # Reshape into grid patches and compute all means/stds at once
        grid = img_gpu[:GRID_SIZE*gh, :GRID_SIZE*gw].reshape(GRID_SIZE, gh, GRID_SIZE, gw)
        grid_means = cp.mean(grid, axis=(1, 3)).flatten().get().tolist()
        grid_stds = cp.std(grid, axis=(1, 3)).flatten().get().tolist()

        # --- SYNC TO CPU FOR COMPLEX LOGIC ---
        # We do Skeleton and Connected Components on CPU as they aren't natively 'mathy'
        img_low_cpu = mask_low.get()
        img_high_cpu = mask_high.get()
        
        skel = morphology.skeletonize(img_low_cpu)
        skeleton_length = np.sum(skel)

        labels = measure.label(img_high_cpu)
        props = measure.regionprops(labels)
        num_clusters = len(props)
        mean_cluster_area = np.mean([p.area for p in props]) if props else 0
        mean_ecc = np.mean([p.eccentricity for p in props]) if props else 0

        # Stats
        pix_crys = img_cpu[img_cpu < THRES]
        if pix_crys.size > 2:
            m_int, s_int = np.mean(pix_crys), np.std(pix_crys)
            skew_int, kurt_int = skew(pix_crys), kurtosis(pix_crys)
        else:
            m_int = s_int = skew_int = kurt_int = 0

        # =========================
        # COMPILE RESULTS
        # =========================
        feature_vector = [
            filename, label, area_fraction, num_clusters, 
            mean_cluster_area, mean_ecc, m_int, s_int, 
            skew_int, kurt_int, skeleton_length, mean_thickness, 
            max_thickness, mean_l_std, std_l_std, f_dim, fft_energy
        ] + grid_means + grid_stds

        results.append(feature_vector)
        print(f"GPU/CPU Hybrid: {filename} processed in folder {root}.")

# Save to CSV

#TODO: make it use feature names in headers; make sure all features used in main are in here
pd.DataFrame(results).to_csv("riboflavin_features_gpu.csv", index=False)
print("Data processed.")

TypeError: extract_differentiable_features() missing 1 required positional argument: 'img_tensor'